In [326]:
# Import libraries
import pandas as pd
import numpy as np

In [327]:
# Load data
path = "D:/D/Projects/Healthcare Analytics/data/raw"

admissions = pd.read_csv(path+'/admissions.csv')
claims = pd.read_csv(path+'/claims.csv')
patients = pd.read_csv(path+'/patients.csv')
readmissions = pd.read_csv(path+'/readmissions.csv')

CREATE DATA QUALITY FLAGS

In [328]:
# Convert data column firt, try to do it first
admissions['admit_date'] = pd.to_datetime(admissions['admit_date'])
admissions['discharge_date'] = pd.to_datetime(admissions['discharge_date'])

In [329]:
# Start generate flag for los, dignosis, billing, claims
# Flag LOS data, and count negative day stay because it is impossible dtays
admissions['negative_los_flag'] = admissions['los_days']<0
print('Negative LOS count: ', (admissions['los_days']<0).sum())

# Date Logic error (litle different than negative los)
admissions['date_logic_error_flag'] = (admissions['discharge_date'] < admissions['admit_date'])
print(admissions['date_logic_error_flag'].sum())

# Try to check LOS mismatch
computed_los = (
    admissions['discharge_date'] - admissions['admit_date']
).dt.days

admissions['los_mismatch_flag'] = (
    admissions['los_days'] != computed_los
)
print("LOS mismatch count: ", admissions['los_mismatch_flag'].sum())

Negative LOS count:  2000
2000
LOS mismatch count:  3974


In [330]:
# Orphan patient records
admissions['orphan_patient_flag'] = (
    ~admissions['patient_id'].isin(patients['patient_id'])
)

print("Orphan admissions count: ", admissions['orphan_patient_flag'].sum())

Orphan admissions count:  1000


In [331]:
# This enables:

# filtering safe datasets
# ML-ready datasets
# dashboard-safe subsets
# audit transparency
# reproducibility

# This is enterprise analytics thinking

FINANCIAL INTEGRITY VIOLATION

In [332]:
# Flag for billed amount and approve amount
claims['approved_exceeds_billed_flag'] = (
    claims['approved_amount'] > claims['billed_amount']
)
print("Approved Amount Flag Count: ", claims['approved_exceeds_billed_flag'].sum())

# Payment timeline violation
claims['submission_date'] = pd.to_datetime(claims['submission_date'])
claims['payment_date'] = pd.to_datetime(claims['payment_date'])

claims['payment_before_submission_flag'] = (
    claims['submission_date'] > claims['payment_date']
)
print("Paayment before claim submission count: ", claims['payment_before_submission_flag'].sum())


Approved Amount Flag Count:  4967
Paayment before claim submission count:  501


In [333]:
# Missing payment date when status is approved
claims['missing_payment_date_flag'] = (
    (claims['claim_status'] == "Approved") &
    (claims['payment_date'].isna())
)
print("Missing payment date for approved claims count: ", claims['missing_payment_date_flag'].sum())

# Payment exist but claim not approved
claims['payment_without_approval_flag'] = (
    claims["payment_date"].notna() &
    ~claims["claim_status"].isin(["Approved", "Partial"])
)
print("Payment Exist even claims pending count: ", claims['payment_without_approval_flag'].sum())

# Negative amount in billed
claims['negative_amount_flag'] = (
    (claims['billed_amount'] < 0) |
    (claims['approved_amount'] < 0)
)
print("Negative finacial count: ", claims['negative_amount_flag'].sum())

Missing payment date for approved claims count:  0
Payment Exist even claims pending count:  1966
Negative finacial count:  2000


In [334]:
# Claims flaged columns
claims[[
    "approved_exceeds_billed_flag",
    "payment_before_submission_flag",
    "negative_amount_flag",
    "missing_payment_date_flag",
    "payment_without_approval_flag"
]].sum()

approved_exceeds_billed_flag      4967
payment_before_submission_flag     501
negative_amount_flag              2000
missing_payment_date_flag            0
payment_without_approval_flag     1966
dtype: int64

DETERMINISTIC REPAIR (FIRST REAL CLEANING STEP)

In [335]:
# RULE
# Repair what can be repaired safely, flag what cannot be repaired.

In [336]:
# Repair LOS - never drop this step.
admissions['computed_los'] = (
    admissions['discharge_date'] - admissions['admit_date']
).dt.days

# repair los data
admissions['los_days_corrected'] = admissions['los_days']
admissions.loc[
    admissions['los_mismatch_flag'],
    'los_days_corrected'
] = admissions['computed_los']

# Analytics-safe LOS field
admissions["los_days_analytics"] = admissions["computed_los"]

admissions.loc[
    admissions["date_logic_error_flag"],
    "los_days_analytics"
] = np.nan

print("Negative analytics LOS:", (admissions["los_days_analytics"] < 0).sum())
print("Missing analytics LOS:", admissions["los_days_analytics"].isna().sum())

Negative analytics LOS: 0
Missing analytics LOS: 2000


In [337]:
# Repairing Diagnosis code
# Preserve original diagnosis code
admissions['diagnosis_code_original'] = admissions['diagnosis_code']
admissions['diagnosis_desc_original'] = admissions['diagnosis_desc']

# Perform standardization on diagnosis data to clea it. 
icd_pattern = r"^[A-Z][0-9]{2}(\.[A-Z0-9]{1,3})?$"

# Clean diagnosis code formating only
admissions['diagnosis_code_cleaned'] = (
    admissions['diagnosis_code']
    .astype("string")
    .str.strip()
    .str.upper()
)

# Clean diagnosis description formating
admissions['diagnosis_desc_cleaned'] = (
    admissions['diagnosis_desc']
    .astype("string")
    .str.strip()
)

# missing diagnosis codes with ICD format check
admissions['diagnosis_missing_flag'] = admissions['diagnosis_code'].isna()


admissions['diagnosis_invalid_format_flag'] = (
    admissions['diagnosis_code'].notna() &
    ~admissions['diagnosis_code_cleaned'].str.match(icd_pattern, na=False)
)

# Suspicious diagnosis description
admissions['diagnosis_desc_invalid_flag'] = (
    admissions['diagnosis_desc']
    .astype("string")
    .str.strip()
    .str.upper()
    .isin(["INVALID", "N/A", "MISSING", ""])
)

# create diagnosis usable flag for analytics
admissions["diagnosis_usable_flag"] = ~(
    admissions["diagnosis_missing_flag"] |
    admissions["diagnosis_invalid_format_flag"] |
    admissions["diagnosis_desc_invalid_flag"]
)
print("Diagnosis usable row: ", admissions['diagnosis_usable_flag'].sum())
print("Diagnosis unusable row: ", (~admissions['diagnosis_usable_flag']).sum())


Diagnosis usable row:  196000
Diagnosis unusable row:  4000


In [338]:
# Final flagged data
admissions[
[
"negative_los_flag",
"date_logic_error_flag",
"los_mismatch_flag",
"diagnosis_missing_flag",
"diagnosis_invalid_format_flag",
"diagnosis_desc_invalid_flag",
"orphan_patient_flag"
]
].sum()

negative_los_flag                2000
date_logic_error_flag            2000
los_mismatch_flag                3974
diagnosis_missing_flag            512
diagnosis_invalid_format_flag    3000
diagnosis_desc_invalid_flag      4000
orphan_patient_flag              1000
dtype: Int64

In [339]:
# Orphan admission handling
admissions['patient_link_usable_flag'] = ~admissions['orphan_patient_flag']

print("Linkable admissions:", admissions["patient_link_usable_flag"].sum())
print("Non-linkable admissions:", (~admissions["patient_link_usable_flag"]).sum())

Linkable admissions: 199000
Non-linkable admissions: 1000


In [340]:
# WHO to update           WHAT column        WHAT value to put in
# ─────────────────────   ──────────────     ───────────────────
# claims.loc[flag==True,  "approved_          = claims["billed_
#                          amount_corrected"]    amount"]

In [341]:
# Claim file rearing beging
claims['billed_amount_original'] = claims['billed_amount']
claims['aprroved_amount_original'] = claims['approved_amount']

# create corrected aprroved amount
claims['approved_amount_corrected'] = claims['approved_amount']
claims.loc[
    claims['approved_exceeds_billed_flag'],
    'approved_amount_corrected'
] = claims['billed_amount']

# Count mismatch rows afer cleannign
print((
    claims['approved_amount_corrected'] > claims['billed_amount']
).sum())

# Hnadle negative finacial values safely
claims['finacial_amount_usable_flag'] = ~claims['negative_amount_flag']
print("Usable finacial rows: ", claims['finacial_amount_usable_flag'].sum())

0
Usable finacial rows:  198000


In [342]:
# Normalization of the claim status
claims['claim_status_original'] = claims['claim_status']

claims['claim_status_cleaned'] = (
    claims['claim_status']
    .astype("string")
    .str.strip()
    .str.upper()
    .str.title()
)

claims['claim_status_cleaned'].value_counts()

claim_status_cleaned
Approved    98245
Denied      39235
Pending     38966
Partial     19554
Name: count, dtype: Int64

In [343]:
# Normalization of payer
claims['payer_original'] = claims['payer']

claims['payer_cleaned'] = (
    claims['payer']
    .astype("string")
    .str.strip()
    .str.upper()
    .str.title()
)

claims['payer_cleaned'].value_counts()

payer_cleaned
Medicaid        28692
Cigna           28630
Aetna           28608
Medicare        28591
Self-Pay        28582
Unitedhealth    28532
Bluecross       28365
Name: count, dtype: Int64

In [344]:
# Create claims financial usability flag
claims['financial_record_usable_flag'] = ~(
    claims['negative_amount_flag'] | claims['approved_exceeds_billed_flag']
)
claims['financial_record_usable_flag'].value_counts()

financial_record_usable_flag
True     195033
False      4967
Name: count, dtype: int64

In [345]:
# Create claim workflow usable flag
claims['workflow_record_usable_flag'] = ~(
    claims['payment_before_submission_flag'] |
    claims['payment_without_approval_flag'] |
    claims['missing_payment_date_flag']
)
claims['workflow_record_usable_flag'].value_counts()

workflow_record_usable_flag
True     197542
False      2458
Name: count, dtype: int64

In [346]:
admissions["admission_record_analytics_ready_flag"] = (
    admissions["diagnosis_usable_flag"] &
    admissions["patient_link_usable_flag"] &
    admissions["los_days_analytics"].notna()
)
admissions["admission_record_analytics_ready_flag"].value_counts()

admission_record_analytics_ready_flag
True     193072
False      6928
Name: count, dtype: Int64

In [347]:
admissions_ready = admissions[
    admissions["admission_record_analytics_ready_flag"]
].copy()

print("Analytics-ready admissions:", len(admissions_ready))

Analytics-ready admissions: 193072


In [348]:
claims["claim_record_analytics_ready_flag"] = (
    claims["financial_record_usable_flag"] &
    claims["workflow_record_usable_flag"]
)

print("Analytics-ready claims:",
      claims["claim_record_analytics_ready_flag"].sum())

print(claims["claim_record_analytics_ready_flag"].value_counts())

Analytics-ready claims: 192638
claim_record_analytics_ready_flag
True     192638
False      7362
Name: count, dtype: int64


EXPORT CLEAN ANALYTICS DATASETS WITH ORIGINAL DATA

In [349]:
# # Export dataset
# cleaned_dataset_path = "D:/D/Projects/Healthcare Analytics/data/clean"
# admissions.to_csv(cleaned_dataset_path+'/admissions_clean.csv', index=False)
# claims.to_csv(cleaned_dataset_path+'/claims_clean.csv', index=False)
# patients.to_csv(cleaned_dataset_path+'/patients_clean.csv', index=False)
# readmissions.to_csv(cleaned_dataset_path+'/readmissions_clean.csv', index=False)


PHASE 4: CLAIM STATUS DISTRIBUTION

In [350]:
# Let's build KPI for every claims
claims['claim_status_cleaned'].value_counts()

claim_status_cleaned
Approved    98245
Denied      39235
Pending     38966
Partial     19554
Name: count, dtype: Int64

In [351]:
#  Clculating Approval rate, denial rate, pending rate
approval_rate = round(
    (claims['claim_status_cleaned'] == "Approved").mean() * 100,
    2
)

pending_rate = round(
    (claims['claim_status_cleaned'] == "Pending").mean() * 100,
    2
)

denial_rate = round(
    (claims['claim_status_cleaned'] == "Denied").mean() * 100,
    2
)

partial_rate = round(
    (claims['claim_status_cleaned'] == "Partial").mean() * 100,
    2
)

print("Approval rate:", approval_rate)
print("Denial rate:", denial_rate)
print("Pending rate:", pending_rate)
print("Partial rate:", partial_rate)


Approval rate: 50.12
Denial rate: 20.02
Pending rate: 19.88
Partial rate: 9.98


In [352]:
# Total value for payes
claims['payer_cleaned'].value_counts()

payer_cleaned
Medicaid        28692
Cigna           28630
Aetna           28608
Medicare        28591
Self-Pay        28582
Unitedhealth    28532
Bluecross       28365
Name: count, dtype: Int64

In [353]:
# Approval rate based on payer
claims.groupby('payer_cleaned')['claim_status_cleaned'] \
        .apply(lambda x: (x == 'Approved').mean()) \
        .round(3)

payer_cleaned
Aetna           0.500
Bluecross       0.499
Cigna           0.503
Medicaid        0.502
Medicare        0.501
Self-Pay        0.501
Unitedhealth    0.504
Name: claim_status_cleaned, dtype: float64

In [354]:
# Denial reasons ditribution
claims['denial_reason'].value_counts()

denial_reason
Duplicate Claim            6768
Missing Information        6700
Not Medically Necessary    6694
Out of Network             6691
Authorization Required     6650
Expired Coverage           6534
Name: count, dtype: int64

In [355]:
# Financial Exposure Mettrics
claims[["approved_amount_corrected", "billed_amount"]].describe()

,approved_amount_corrected,billed_amount
count,200000.000000,200000.000000
mean,42849.638353,73646.800690
std,39140.239926,45726.093845
min,-149979.980000,-149979.980000
25%,6315.490000,36563.795000
50%,39195.370000,74349.205000
75%,72488.335000,112132.832500
max,149977.020000,149998.070000


In [356]:
# Compute approval efficient ratio
approval_ratio = (
    claims['approved_amount'].sum() /
    claims['billed_amount'].sum()
)

print('Overall payout ratio: ', round(approval_ratio, 3))

Overall payout ratio:  0.601


In [357]:
# Create analytics-safe financial columns
claims["approved_amount_analytics"] = claims["approved_amount"].where(
    claims["financial_record_usable_flag"]
)

claims["billed_amount_analytics"] = claims["billed_amount"].where(
    claims["financial_record_usable_flag"]
)

# Repeat analysis using only analytics - ready claims
claims_ready = claims[
    claims['claim_record_analytics_ready_flag']
].copy()

claims_ready['claim_status_cleaned'].value_counts()

claims_ready.groupby('payer_cleaned')['claim_status_cleaned'] \
            .apply(lambda x: (x == "Approved").mean()) \
            .round(3)

payer_cleaned
Aetna           0.499
Bluecross       0.497
Cigna           0.501
Medicaid        0.500
Medicare        0.500
Self-Pay        0.500
Unitedhealth    0.502
Name: claim_status_cleaned, dtype: float64

In [358]:
# Denial rate by payer ready
denial_by_payer_ready = (
    claims_ready.groupby('payer_cleaned')['claim_status_cleaned'] \
                .apply(lambda x: (x == "Denied").mean()) \
                .round(3)
)

print(denial_by_payer_ready)

payer_cleaned
Aetna           0.202
Bluecross       0.201
Cigna           0.200
Medicaid        0.199
Medicare        0.199
Self-Pay        0.204
Unitedhealth    0.198
Name: claim_status_cleaned, dtype: float64


In [ ]:
# Payout efficiency ratio by payer
payout_ratio_by_payer = (
    claims_ready
    .groupby('payer_cleaned')
    .apply(lambda x: x["approved_amount_analytics"].sum()
                     / x["billed_amount_analytics"].sum())
    .round(3)
)

print(payout_ratio_by_payer)

In [ ]:
print("Average insurer payout ratio ranges between 57%–58% of billed amounts across payers.")
print("Uniform distribution suggests settlement behavior is not payer-biased in synthetic dataset.")

Average insurer payout ratio ranges between 57%–58% of billed amounts across payers.
Uniform distribution suggests settlement behavior is not payer-biased in synthetic dataset.


In [ ]:
# KPI for denial reason distribution
claims_ready[claims_ready['claim_status_cleaned'] == 'Denied'] \
            ['denial_reason'] \
            .value_counts(normalize = True) \
            .round(3)

denial_reason
Duplicate Claim            0.168
Not Medically Necessary    0.167
Missing Information        0.167
Out of Network             0.167
Authorization Required     0.167
Expired Coverage           0.164
Name: proportion, dtype: float64

In [ ]:
# Workflow issue summery based on payment before submission and payment without approval_rate
workflow_issue_summary = claims_ready[
    ["payment_before_submission_flag",
    "payment_without_approval_flag"]
].sum()

print(workflow_issue_summary)

payment_before_submission_flag    0
payment_without_approval_flag     0
dtype: int64


In [ ]:
# Final KPI for claim settlement completion rate
settled_claim_rate = (
    claims_ready['claim_status_cleaned']
    .isin(['Approved', 'Denied'])
    .mean()
    .round(3)
)

print("Final settled claim rate: ", settled_claim_rate)

Final settled claim rate:  0.693


In [ ]:
# Export the claim ready file.
# claims_ready.to_excel(cleaned_dataset_path + '/claims_ready.xlsx', index = False)

PHASE 6: READMISSIONS ANALYSIS

In [ ]:
# check data of readmissions
print(readmissions.shape)
readmissions.head(5)

(3000, 8)


,readmit_id,patient_id,original_admission_id,readmit_date,days_since_discharge,readmit_reason,readmit_department,readmit_severity
0,RDM-00001,PAT-05125,ADM-177898,2022-10-27,11,Unrelated Condition,ICU,Critical
1,RDM-00002,PAT-06011,ADM-115928,2023-05-07,29,Follow-up Required,General Surgery,High
2,RDM-00003,PAT-08235,ADM-182515,2021-10-30,21,Infection,Oncology,Low
3,RDM-00004,PAT-03914,ADM-134681,2022-03-06,7,Follow-up Required,Neurology,Medium
4,RDM-00005,PAT-02488,ADM-039352,2022-08-12,8,Infection,Oncology,Low


In [ ]:
# Validate readmission analytics
# check readmission's admisison data available in admission file
valid_admission_links = readmissions['original_admission_id'].isin(admissions['admission_id'])

print("Valid admisison links: ", valid_admission_links.sum())
print("Invalid admisison links: ", (~valid_admission_links).sum())

Valid admisison links:  3000
Invalid admisison links:  0


In [ ]:
# Validate 40 day rule integrity
invalid_readmit_days = (
    (readmissions['days_since_discharge'] < 1) |
    (readmissions['days_since_discharge'] > 30)
)

print("Invalid Readmission day values: ", invalid_readmit_days.sum())

Invalid Readmission day values:  0


In [ ]:
# Check patient id data in readmissions
readmit_valid_patient_link_flag = readmissions['patient_id'].isin(patients['patient_id'])

print("Valid Patient links: ", readmit_valid_patient_link_flag.sum())
print("InValid Patient links: ", (~readmit_valid_patient_link_flag).sum())

Valid Patient links:  2981
InValid Patient links:  19


In [ ]:
# There are 19 invalid patient, that's means there are some patient's data exist which are not in patient data. SO we generate flag as like in admissions
readmissions['orphan_readmit_patient_flag'] = ~readmissions['patient_id'].isin(patients['patient_id'])
readmissions['orphan_admission_patient_flag'] = ~readmissions['patient_id'].isin(admissions['patient_id'])

print("Orphan readmission patient count: ", readmissions['orphan_readmit_patient_flag'].sum())
print("Orphan admission patient count: ", readmissions['orphan_admission_patient_flag'].sum())

Orphan readmission patient count:  19
Orphan admission patient count:  0


In [ ]:
# Create analytics ready readmission flag
readmissions['readmission_record_analytics_ready_flag'] = ~(
    readmissions['orphan_readmit_patient_flag'] |
    readmissions['orphan_admission_patient_flag']
)
print(readmissions['readmission_record_analytics_ready_flag'].value_counts())

readmission_record_analytics_ready_flag
True     2981
False      19
Name: count, dtype: int64


In [ ]:
# Create analytics-ready subset
readmissions_ready = readmissions[readmissions['readmission_record_analytics_ready_flag']]
print("Analytics-ready readmissions:", len(readmissions_ready))

Analytics-ready readmissions: 2981


In [ ]:
# Generate analysis rate for readmissions_ready
readmission_rate = len(readmissions_ready) / len(admissions_ready)
print("30-day readmission rate:", round(readmission_rate, 4))
print("30-day readmission rate:", round(readmission_rate * 100, 2), "%")
print("Synthetic dataset produces a lower-than-realistic readmission rate.")
print("In real hospital populations this metric typically ranges between 8% and 15%.")

30-day readmission rate: 0.0154
30-day readmission rate: 1.54 %
Synthetic dataset produces a lower-than-realistic readmission rate.
In real hospital populations this metric typically ranges between 8% and 15%.


In [ ]:
readmit_by_department = (
    readmissions_ready['readmit_department']
    .value_counts(normalize = True)
    .round(3)
)

print(readmit_by_department)

readmit_department
Cardiology         0.134
Oncology           0.131
Pediatrics         0.127
General Surgery    0.127
Neurology          0.124
Orthopedics        0.120
Emergency          0.118
ICU                0.118
Name: proportion, dtype: float64


In [ ]:
readmissions_ready = readmissions_ready.merge(
    admissions_ready[['admission_id', 'department']],
    left_on = 'original_admission_id',
    right_on = 'admission_id',
    how = 'left'
)
readmissions_ready

,readmit_id,patient_id,original_admission_id,readmit_date,days_since_discharge,readmit_reason,readmit_department,readmit_severity,orphan_readmit_patient_flag,orphan_admission_patient_flag,readmission_record_analytics_ready_flag,admission_id,department
0,RDM-00001,PAT-05125,ADM-177898,2022-10-27,11,Unrelated Condition,ICU,Critical,False,False,True,NaN,NaN
1,RDM-00002,PAT-06011,ADM-115928,2023-05-07,29,Follow-up Required,General Surgery,High,False,False,True,ADM-115928,Orthopedics
2,RDM-00003,PAT-08235,ADM-182515,2021-10-30,21,Infection,Oncology,Low,False,False,True,ADM-182515,Oncology
3,RDM-00004,PAT-03914,ADM-134681,2022-03-06,7,Follow-up Required,Neurology,Medium,False,False,True,ADM-134681,Pediatrics
4,RDM-00005,PAT-02488,ADM-039352,2022-08-12,8,Infection,Oncology,Low,False,False,True,ADM-039352,Oncology
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2976,RDM-02996,PAT-09645,ADM-068666,2021-02-17,30,Infection,General Surgery,Low,False,False,True,ADM-068666,Orthopedics
2977,RDM-02997,PAT-07383,ADM-077279,2023-12-14,22,Unrelated Condition,Oncology,Low,False,False,True,ADM-077279,ICU
2978,RDM-02998,PAT-07266,ADM-005191,2021-06-25,5,Medication Issue,Cardiology,Medium,False,False,True,ADM-005191,Orthopedics
2979,RDM-02999,PAT-07694,ADM-054119,2020-06-21,16,Medication Issue,Oncology,Medium,False,False,True,ADM-054119,Pediatrics
